### Integration of Open3dsg and Llava-3D


#### **Section 1: Introduction**

Objective:
- Integrate the outputs of Open3DSG as input to the LLM of LLaVA 3D.
- Process and structure relevant information: objects, relationships, spatial positions, and camera pose.

Flow:
1. Extract relevant data from Open3DSG.
2. Enrich the information with textual descriptions.
3. Prepare a structured text input that LLaVA 3D can process.

In [1]:
# Imports
%load_ext autoreload
%autoreload 2

import sys
sys.path.append( '/Volumes/scratch/alegretelena/LLaVA-3D/open3dsg' )


import os
import json
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

from const import CONF_PATH_R3SCAN_RAW
from preprocess_3rscan import Preprocessor
from open_dataset import Open2D3DSGDataset
from get_object_frame import run, read_json


# Global Variables
objects_json = relationships_json = relationships_dict = scan_data = classes_dict = None

/Users/elenaalegretregalado/miniconda3/envs/lap_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'open3dsg'

##### Extracting data from Open3dsg

1. `Objects.json`: Contains information about objects in the 3D scene. For each scan includes:  ==> Not info about rel, just obj
- affordances, colors, and semantic properties.
- Structure:
    - object_json = {['scans']}
        - object_json['scans'] = [..., {'scan': info1,  'objects': info2}, ...] list of dicts
            - info1 = id
            - info2 = ['ply_color', 'nyu40', 'eigen13', 'label', 'rio27', 'affordances', 'id', 'global_id', 'attributes']

2. `relationships.json`: Contains information about relationships between two objects in the same 3D scene. 
- Structure: 
    - relationships_json = {['scans']}
        - relationships_json['scans'] =  [..., {'scan': info1,  'relationships': info2}, ...] list of dicts
            - info1 = id
            - info2 = [..., [id_obj1, id_obj2, id_rel, text_rel], ...]

3. `relationships.txt`: Contains correspondance between text and id_rel. 
-  line_idx = id_rel == text_rel

4. `classes.txt`:  Contains correspondance between text and id_obj. 
-  line_idx = id_obj == text_obj


##### Extracting data from Llava-3D

`embodiedscan_infos_full.json`: Contains information about each scene. For each scene, contains information (pose, depth) about the images that belongs to it.
- IMP: Has info about scannet and 3rscan

- Structure: 
    - FOR SCANNET:
        - scan_data = {..., 'scannet/scene0191_00': info1, ...}
            - (info1) scan_data['scannet/scene0191_00'] = {..., 'scannet/posed_images/scene0191_00/00000.jpg': info2, ...}
                - (info2) scan_data['scannet/scene0191_00']['scannet/posed_images/scene0191_00/00000.jpg'] = {'pose': [num], 'depth': 'scannet/posed_images/scene0191_00/00000.png'}

    - FOR 3DSCAN: 
        - scan_data['3rscan/3rscan0002']['3rscan/02b33dfb-be2b-2d54-92d2-cd012b2b3c40/sequence/frame-000000.color.jpg']


In [2]:
# Read the files if not already read
if not objects_json:
    with open("../data/3RScan/objects.json") as file:
        objects_json = json.load(file)

if not relationships_json:
    with open("../data/3RScan/relationships.json") as file:
        relationships_json = json.load(file)

if not relationships_dict:
    relationships_dict = {}
    with open('../data/3RScan/relationships.txt') as file:
        for idx, x in enumerate(file):
            relationships_dict[idx] = x.split('\n')[0]

if not classes_dict:
    classes_dict = {}
    with open('../data/3RScan/classes.txt') as file:
        for idx, x in enumerate(file):
            classes_dict[idx] = x.split('\n')[0]

if not scan_data:
    with open("../data/embodiedscan_infos_full.json") as file:
        scan_data = json.load(file)

### Analysis for 3Dscan dataset inside embodiedscan_infos_full.json

In [ ]:
# Define the value to find
VALUETOFIND = '754e884c-ea24-2175-8b34-cead19d4198d'

# Function to search for the value in the dictionary
def find_value_in_scan_data(scan_data, valuetofind):
    matches = set()
    for key1, sub_dict in scan_data.items():
        if key1.startswith('3rscan') and isinstance(sub_dict, dict):
            for key2 in sub_dict.keys():
                # Only process keys containing '/'
                if '/' in key2:
                    elements = key2.split('/')
                    if len(elements) > 1:# and elements[1] == valuetofind:
                        #matches.append(f"Found match in {key1}/{key2}")
                        matches.add(elements[1])

    return matches

# Perform the search
matches = find_value_in_scan_data(scan_data, VALUETOFIND)

# Output the results
if matches:
    print(matches)
else:
    print(f"No matches found for {VALUETOFIND}.")

In [ ]:
# Print an img of the matched scan
import matplotlib.pyplot as plt
import os
from PIL import Image

# Define the file path to the image
base_path = "../data/"  # Replace with the base directory of your dataset
image_path = os.path.join(base_path, "3RScan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000000.color.jpg")

# Load and display the image
if os.path.exists(image_path):
    img = Image.open(image_path)  # Open the image
    plt.figure(figsize=(10, 6))   # Set the figure size
    plt.imshow(img)               # Display the image
    plt.axis('off')               # Hide axes for better visualization
    plt.title("Matched Image")    # Add a title
    plt.show()
else:
    print(f"Image not found at: {image_path}")

## Section 2: Integrate the code

`GOAL`: Given a relationship, find the images where the two objects creating the relationship are visible. 
- data_dict['obj2frame'] ==> Relates objescts with images. 
- data_dict['triples'] ==> Relates objects and relationships.


In [6]:
# Functions

def load_scan(base_path, file_path):
    return json.load(open(os.path.join(base_path, file_path)))["scans"]

def obtain_frames(obj1_frames, obj2_frames, data_dict):    
    """
    Filters and finds common frames between two sets of object frames.
        :param obj1_frames (list): List of tuples representing frames for object 1.
        :param obj2_frames (list): List of tuples representing frames for object 2.
        
    Returns (list): A list of common frame names (strings) between the two objects.
    """
    if '3rscan' in data_dict['dataset']:
        path = '../../data' + data_dict['dataset'] +'/'+ data_dict['scene_id'] +'/sequence/'
    elif 'scannet' in data_dict['dataset']:
        path = '../../data' + data_dict['dataset'] +'/'+ data_dict['scene_id'] +'/color/'

    # Filter frames starting with 'frame' for both objects
    frames_obj1 = [path+frame[0] for frame in obj1_frames if isinstance(frame, tuple) and isinstance(frame[0], str) and frame[0].startswith('frame')]
    frames_obj2 = [path+frame[0] for frame in obj2_frames if isinstance(frame, tuple) and isinstance(frame[0], str) and frame[0].startswith('frame')]
     
    # Find common frames
    common_frames = np.intersect1d(frames_obj1, frames_obj2)
    return common_frames.tolist()

def obtain_the_common_images(data_dict): 
    """
    Processes relationships between objects and finds common frames for each pair of related objects.
        :param data_dict (dict): A dictionary containing:
            - 'triples' (list): A list of object relationships in the format [obj1, relation, obj2].
            - 'obj2frame' (dict): A dictionary mapping object IDs to their respective frame lists.
    
    Returns common_frames (list): A list of common frames for each relationship, stored in `data_dict['common_frames']`.
    """
    # Validate that required keys exist
    if 'triples' not in data_dict or 'obj2frame' not in data_dict:
        raise KeyError("The dictionary must contain the keys 'triples' and 'obj2frame'.")
    
    relationships = data_dict['triples']
    data_dict['common_frames'] = []  # Initialize the list of common frames
    
    for relationship in relationships:
        obj1 = relationship[0]
        obj2 = relationship[1]
        
        # Validate that objects exist in obj2frame
        if obj1 not in data_dict['obj2frame'] or obj2 not in data_dict['obj2frame']:
            print(f"Warning: {obj1} or {obj2} not found in 'obj2frame'.")
            data_dict['common_frames'].append([])
            continue
        
        obj1_frames = data_dict['obj2frame'][obj1]
        obj2_frames = data_dict['obj2frame'][obj2]
        
        # Obtain common frames and add them
        common_frames = obtain_frames(obj1_frames, obj2_frames, data_dict)
        data_dict['common_frames'].append(common_frames)
    
    print("Processing completed. Common frames are stored in 'common_frames'.")
    return data_dict

IMPORTANT: 
- scene_id: A unique identifier for a 3D scene.
    - P.e: '754e884c-ea24-2175-8b34-cead19d4198d' 
    
- scan_id: A unique identifier for a specific scan or instance of a scene.
    - P.e: '754e884c-ea24-2175-8b34-cead19d4198d-3'


COMMENT: 
- In order to map our 3RSCAN dataset () with the 3RSCAN dataset used in Llava-3D we use `embodiedscan_infos_full.json`. 

    - Our dataset structure: 
    

In [7]:
if __name__ == '__main__':
    scene_id = '754e884c-ea24-2175-8b34-cead19d4198d'        # If you want to process thw whole dataset, delete this line

    D3SSG = load_scan(CONF_PATH_R3SCAN_RAW, "relationships_train.json")
    
    for r in D3SSG:
        if r['scan'] == scene_id:
            D3SSG = [r]

    dataset = Open2D3DSGDataset(
        relationships_R3SCAN=D3SSG,
        relationships_scannet=None,
        openseg=False,
        img_dim=224,
        rel_img_dim=224,
        top_k_frames=5,
        scales=3,
        mini=False,
        load_features=None,
        blip=True,
        llava=False,
        half=False,
        max_objects=9,
        max_rels=72
    )

    # Process relationships in the dataset
    for instance in dataset: 
        print(dataset)
        data_dict = obtain_the_common_images(instance)

100%|██████████| 1/1 [00:13<00:00, 13.75s/it]


Processing completed. Common frames are stored in 'common_frames'.


In [8]:
data_dict['common_frames'][0]

['../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000090.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000100.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000110.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000230.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000270.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000420.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000460.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000740.color.jpg']

In [27]:
data_dict['common_frames'][0]

['../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000090.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000100.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000110.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000230.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000270.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000420.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000460.color.jpg',
 '../../data3rscan/754e884c-ea24-2175-8b34-cead19d4198d/sequence/frame-000740.color.jpg']

In [26]:
scan_data['scannet/scene0356_00']['scannet/posed_images/scene0356_00/00000.jpg']

{'pose': [[-0.270071, 0.36267, -0.891926, 2.757273],
  [0.962506, 0.077249, -0.260031, 3.732496],
  [-0.025405, -0.928711, -0.369935, 1.491022],
  [0.0, 0.0, 0.0, 1.0]],
 'depth': 'scannet/posed_images/scene0356_00/00000.png'}

In [28]:
scan_data['3rscan/3rscan0002']['3rscan/02b33dfb-be2b-2d54-92d2-cd012b2b3c40/sequence/frame-000000.color.jpg']

{'pose': [[0.0944584, -0.984421, -0.1483, -0.0434051],
  [-0.744943, -0.168714, 0.645442, 0.0326288],
  [-0.660407, 0.0495076, -0.749274, -0.137367],
  [0.0, 0.0, 0.0, 1.0]],
 'depth': '3rscan/02b33dfb-be2b-2d54-92d2-cd012b2b3c40/sequence/frame-000000.depth.pgm'}

## TRY

In [ ]:
def preprocess_embodiedscan_infos_full(scan_data): 
    for scan_id, scan_dict in scan_data.items():
        # Preprocessing for 3RScan
        if scan_id.startswith('3rscan') and isinstance(scan_dict, dict):
            for key2 in scan_dict.keys():
                # Only process keys containing '/'
                if '/' in key2:
                    scan_id = scan_id + '/' + key2.split('/')[1] # Obtain the 02b33dfb-be2b-2d54-92d2-cd012b2b3c40 value
                    break
        elif scan_id.startswith('scannet'):
            pass
    
    # Save the updated data to a JSON file
    with open('../data/embodiedscan_infos_full1.json', "w") as outfile:
        json.dump(scan_data, outfile, indent=4)

preprocess_embodiedscan_infos_full(scan_data)

In [ ]:
import json

def preprocess_embodiedscan_infos_full(scan_data): 
    updated_scan_data = {}  

    for scan_id, scan_dict in scan_data.items():
        # Preprocessing for 3RScan
        if scan_id.startswith('3rscan') and isinstance(scan_dict, dict):
            updated_scan_id = scan_id  # Default to the original scan_id
            for key2 in scan_dict.keys():
                # Only process keys containing '/'
                if '/' in key2:
                    updated_scan_id = scan_id + '/' + key2.split('/')[1]  # Update scan_id
                    break
            
            # Store the updated scan_id in the new dictionary
            updated_scan_data[updated_scan_id] = scan_dict
        
        elif scan_id.startswith('scannet'):
            # For scannet, copy data without modification
            updated_scan_data[scan_id] = scan_dict

    # Save the updated data to a JSON file
    with open('../data/embodiedscan_infos_full1.json', "w") as outfile:
        json.dump(updated_scan_data, outfile, indent=4)

    print("Updated scan data saved to '../data/embodiedscan_infos_full1.json'")

# Example usage
preprocess_embodiedscan_infos_full(scan_data)

In [ ]:
# USEFUL
all_images = []  # To store all images across relationships
# objects_json['scans] --> numero llarg i objects[id] numero objecte


for scan_rel in relationships_json['scans']:
    scan_id = scan_rel['scan']                      # Scan identifier
    relationship_scan = scan_rel['relationships']   # List of relationships in the scan


    for scan, scan_dict in scan_data.items():
        if scan.startswith('3rscan') and isinstance(scan_dict, dict):
            # Filter attributes that match the scan_id
            images = [{attr: attr_dict} for attr, attr_dict in scan_dict.items() if str(scan_id) in attr]
            
            # Add these images to the global list
            if images:  # Avoid adding empty lists
                all_images.append({
                    "scan_id": scan_id,
                    "relationships": relationship_scan,
                    "images": images
                })

# Print or process the collected images
print("Collected images for all relationships:")
for entry in all_images:
    print(entry['images'])
    break


In [ ]:
all_images = []  # To store all images across relationships
# objects_json['scans] --> numero llarg i objects[id] numero objecte


for scan_rel in relationships_json['scans']:
    """
    scan_id: Scan identifier 
    relationship_scan
    """
    scan_id = scan_rel['scan']                      # Scan identifier
    relationship_scan = scan_rel['relationships']   # List of relationships in the scan


    for scan, scan_dict in scan_data.items():
        if scan.startswith('3rscan') and isinstance(scan_dict, dict):
            # Filter attributes that match the scan_id
            images = [{attr: attr_dict} for attr, attr_dict in scan_dict.items() if str(scan_id) in attr]
            
            # Add these images to the global list
            if images:  # Avoid adding empty lists
                all_images.append({
                    "scan_id": scan_id,
                    "relationships": relationship_scan,
                    "images": images
                })

# Print or process the collected images
print("Collected images for all relationships:")
for entry in all_images:
    print(entry['images'])
    break


In [ ]:
# Objectiu 1: Donada una relationship, agafar obj1 i obj2 i seleccionar les instancies on els objectes apareixen

#preprocess_embodiedscan_infos_full(scan_data)

for scan_rel in relationships_json['scans']: 
    scan_id = scan_rel['scan']                      # Scan identifier                       P.e: f62fd5fd-9a3f-2f44-883a-1e5cf819608e
    relationship_scan = scan_rel['relationships']   # List of relationships in the scan     P.e: [.. ,[2, 1, 15, 'standing on'], ...]

    for relationship in relationship_scan:
        obj1, obj2 = relationship[0], relationship[1]
        object1 = objects_json['scans'][scan_id]['objects'][obj1]
        # objects_json['scans] --> numero llarg i objects[id] numero objecte
        print('relationship', relationship, obj1, obj2)

        
        
        pass
        for scan, scan_dict in scan_data.items(): 
            if scan.startswith('3rscan') and isinstance(scan_dict, dict):
                images = [{attr: attr_dict} for attr, attr_dict in scan_dict.items() if str(scan_id) in attr]
        print('images', images)
                    
        break
                
            #elif scan_id.startswith('scannet') and isinstance(scan_dict, dict):
            #    pass

            #pass
                


In [ ]:
scan_data['scannet/scene0191_00'].keys()

In [ ]:
scan_data['3rscan/3rscan0002']['3rscan/02b33dfb-be2b-2d54-92d2-cd012b2b3c40/sequence/frame-000000.color.jpg']

In [ ]:
# Check if any key in scan_data starts with "3rscan"
keys_starting_with_3rscan = [key for key in scan_data.keys() if key.startswith('3rscan')]

# Display the keys found
if keys_starting_with_3rscan:
    print("Keys starting with '3rscan':", keys_starting_with_3rscan)
else:
    print("No keys starting with '3rscan' found in scan_data.")


In [ ]:
objects_json['scans'][0]['objects'][0]

# objects_json['scans] --> numero llarg i objects[id] numero objecte

In [ ]:
relationships_json['scans'][0]